In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Configuração de Parâmetros e Caminhos
CATALOGO = "mvp"
ESQUEMA = "staging"
VOLUME = "diabetes"
NOME_ARQUIVO = "diabetes_risk_prediction_dataset.csv"

CAMINHO_CSV = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUME}/{NOME_ARQUIVO}"
TABELA_BRONZE = f"{CATALOGO}.{ESQUEMA}.bronze_diabetes_raw"

# Fixar contexto na sessão
spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")

# Leitura e Adição de Auditoria
df_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(CAMINHO_CSV) \
    .withColumn("_ingestion_datetime", F.current_timestamp()) \
    .withColumn("_source_file", F.lit(CAMINHO_CSV))

# Persistência em Delta Lake
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_BRONZE)

print(f"✓ Camada Bronze criada com sucesso! Linhas: {spark.table(TABELA_BRONZE).count()}")

In [0]:
%sql
-- Validação da tabela Bronze no Databricks SQL
SELECT * FROM mvp.staging.bronze_diabetes_raw LIMIT 10;